# 04 · `uv`: The Complete Guide

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 120 min

> ❗ Not in either source. **uv** (by Astral, the makers of Ruff) has become the most popular Python project manager. It replaces `pip`, `pip-tools`, `pipx`, `virtualenv`, `pyenv`, `twine` and most of Poetry, in one very fast tool written in Rust. This notebook was executed with **uv 0.13**.

### 🎯 What you will learn (every major feature)
1. Installing and updating uv
2. **Python version management**: install, list, find, pin, upgrade; `.python-version`
3. **Running scripts** with inline dependencies (PEP 723), `--with`, shebang scripts
4. **Projects**: `init` (app/lib/package/bare), `add`, `remove`, `sync`, `lock`, `run`, `tree`, `version`, `export`
5. Dependency **groups**, **extras**, **sources** (Git, path, editable, indexes), version **bounds**
6. The **lock file** `uv.lock`: `--locked`, `--frozen`, upgrades
7. **Tools**: `uvx` / `uv tool run|install|upgrade|list|uninstall`
8. The **pip interface**: `uv venv`, `uv pip install|compile|sync|list|tree|freeze`
9. **Workspaces** (monorepos)
10. Code quality: `uv format` (Ruff), `uv check` (ty), `uv audit` (vulnerabilities)
11. `uv build` & `uv publish` (more in lessons 06–07)
12. Configuration (`[tool.uv]`, `uv.toml`), indexes, caching, offline mode, environment variables
13. uv in **Docker** and **CI**; migrating from pip/Poetry; a cheat-sheet

> 🧪 **Sandbox:** the next cell creates a temporary folder and points uv's cache, tool and Python directories into it, so nothing here touches your real setup. It also makes uv use **this notebook's Python** and forbids Python downloads. In your own terminal you'd simply type the `uv ...` commands.

In [1]:
import os, sys, tempfile, pathlib, shutil
LAB = pathlib.Path(tempfile.mkdtemp(prefix="uv-lab-"))
os.environ.update({
    "LAB": str(LAB),
    "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"],  # find the uv installed next to this Python
    "UV_CACHE_DIR": str(LAB / ".uv-cache"),
    "UV_TOOL_DIR": str(LAB / ".uv-tools"),
    "UV_TOOL_BIN_DIR": str(LAB / ".uv-bin"),
    "UV_PYTHON_INSTALL_DIR": str(LAB / ".uv-python"),
    "UV_PYTHON": sys.executable,             # use this notebook's interpreter
    "UV_PYTHON_DOWNLOADS": "never",
    "UV_NO_PROGRESS": "1", "NO_COLOR": "1",
    "GIT_CONFIG_GLOBAL": os.devnull,         # don't read your personal git config (author names/emails)
})
for var in ("FORCE_COLOR", "CLICOLOR_FORCE"):   # plain (uncoloured) command output in the notebook
    os.environ.pop(var, None)
print("uv found at:", shutil.which("uv"))

uv found at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/uv


In [2]:
%%bash
uv --version
uv self version 2>/dev/null || true

uv 0.13.3 (502f93ca6 2026-10-19 aarch64-apple-darwin)


uv 0.13.3 (502f93ca6 2026-10-19 aarch64-apple-darwin)


## 1. Installing uv

```bash
# macOS / Linux (standalone installer)
curl -LsSf https://astral.sh/uv/install.sh | sh
# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
# or with a package manager
brew install uv          # macOS
winget install astral-sh.uv
pipx install uv          # or: pip install uv
```
Keep it current with `uv self update` (standalone installs). Shell completion: `uv generate-shell-completion zsh >> ~/.zshrc`.

### The command map

| Area | Commands |
|---|---|
| Python versions | `uv python install / list / find / pin / upgrade / uninstall / dir` |
| Scripts | `uv run script.py`, `uv init --script`, `uv add --script`, `uv lock --script` |
| Projects | `uv init`, `uv add`, `uv remove`, `uv sync`, `uv lock`, `uv run`, `uv tree`, `uv version`, `uv export` |
| Tools | `uvx`, `uv tool run / install / upgrade / list / uninstall / audit` |
| pip interface | `uv venv`, `uv pip install / uninstall / compile / sync / list / freeze / show / tree / check` |
| Quality | `uv format`, `uv check`, `uv audit` |
| Packaging | `uv build`, `uv publish` |
| Workspaces | `uv workspace list / dir / metadata` |
| Misc | `uv cache clean / prune / dir / size`, `uv auth login`, `uv self update` |

## 2. Python version management

uv can **download and manage Python itself**, so no more pyenv, python.org installers, or "which Python is this?".

```bash
uv python install 3.15            # latest 3.15.x
uv python install 3.12 3.13 3.14  # several at once
uv python install 3.15t           # free-threaded build (no GIL, Module 13)
uv python install pypy@3.11       # PyPy
uv python list                    # available + installed versions
uv python upgrade 3.14            # newest patch release
uv python uninstall 3.12
uv python dir                     # where managed Pythons live
uv python update-shell            # put uv's python on your PATH
```
(Those download Pythons, so they aren't run in this sandbox.) Finding and pinning are safe to run:

In [3]:
%%bash
uv python find                          # which interpreter uv would use
cd "$LAB" && mkdir pinned && cd pinned
uv python pin 3.15                      # writes .python-version
cat .python-version

/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/s

cratchpad/venv315/bin/python


Pinned `.python-version` to `3.15`


3.15


- `.python-version` tells uv (and pyenv) which Python a project uses; uv downloads it automatically if missing.
- Request formats: `3.15`, `3.15.0`, `>=3.12`, `cpython@3.15`, `3.15t`, a path to an interpreter.
- `--python 3.12` on almost any command picks a version for that command.

## 3. Running scripts

### 3.1 A plain script

In [4]:
%%bash
cd "$LAB"
cat > hello.py <<'EOF'
import sys
print(f"Hello from Python {sys.version.split()[0]}")
EOF
uv run hello.py

Hello from Python 3.15.0


### 3.2 Scripts with dependencies: `--with` and inline metadata (PEP 723)

In [5]:
%%bash
cd "$LAB"
cat > pretty.py <<'EOF'
from rich.table import Table
from rich.console import Console
t = Table("Tool", "Replaces")
t.add_row("uv", "pip, venv, pipx, pyenv, pip-tools, twine")
Console(width=70, color_system=None).print(t)
EOF
uv run --with rich pretty.py 2>&1 | grep -v "^Installed\|^Resolved\|^Prepared"     # temporary env with rich

┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━┓
┃ Tool ┃ Replaces                         

        ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━┩
│ uv   │ pip, venv, pipx, pyen

v, pip-tools, twine │
└──────┴─────────────────

─────────────────────────┘


Better: let the script **declare** its dependencies, so anyone can run it with one command:

In [6]:
%%bash
cd "$LAB"
uv init --script fetch.py --python ">=3.12"          # creates the metadata block
uv add --script fetch.py httpx                       # adds a dependency to the block
cat >> fetch.py <<'EOF'

import httpx
print(httpx.get("https://httpbin.org/get", params={"from": "uv"}, timeout=15).json()["args"])
EOF
cat fetch.py
echo "--- running:"
uv run fetch.py 2>&1 | tail -1

Initialized script at `fetch.py`


Resolved 6 packages in 213ms


# /// script
# requires-python = ">=3.15"
# dependencies = [
#     "httpx>=0.28.1",
# ]
# ///


def 

main() -> None:
    print("Hello from fetch.py!")


if __name__ == "__main__":
    main()

import ht

tpx
print(httpx.get("https://httpbin.org/get", params={"from": "uv"}, timeout=15).json()["args"])


--- running:


{'from': 'uv'}


- uv builds a cached, isolated environment for the script; nothing is installed globally.
- `uv lock --script fetch.py` creates `fetch.py.lock` for fully reproducible runs.
- **Executable scripts:** put this shebang on the first line, `chmod +x`, and run `./fetch.py`:
  ```python
  #!/usr/bin/env -S uv run --script
  ```

## 4. Projects

### 4.1 `uv init`: project types

| Command | Creates |
|---|---|
| `uv init myapp` / `--app` | an **application**. In uv 0.13 it's packaged by default: `src/myapp/__init__.py`, a `[project.scripts]` command and the `uv_build` backend |
| `uv init --no-package myapp` | a simple, non-installable app with `main.py` (scripts, notebooks, services) |
| `uv init --lib mylib` | a **library**: src layout, `py.typed`, build backend |
| `uv init --bare` | only `pyproject.toml` |
| `--build-backend hatch\|setuptools\|flit\|pdm\|poetry\|maturin\|scikit\|uv` | choose a backend |
| `--vcs git\|none`, `--no-readme`, `--no-pin-python`, `--author-from auto\|git\|none` | extras |

In [7]:
%%bash
cd "$LAB"
uv init --app --author-from none weather
cd weather
find . -not -path "./.git*" -type f | sort
echo "--------- pyproject.toml"
cat pyproject.toml
echo "--------- src/weather/__init__.py"
cat src/weather/__init__.py

Initialized project `weather` at `/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ea

ljwobe/weather`


./.python-version
./README.md
./pyproject.toml
./src/weather/__init__.py


--------- pyproject.toml


[project]
name = "weather"
version = "0.1.0"
description = "Add your description here"
readme = "REA

DME.md"
requires-python = ">=3.15"
dependencies = []

[project.scripts]
weather = "weather:main"

[b

uild-system]
requires = ["uv_build>=0.13.3,<0.14.0"]
build-backend = "uv_build"


--------- src/weather/__init__.py


def main() -> None:
    print("Hello from weather!")


In [8]:
%%bash
cd "$LAB/weather"
uv run weather            # creates .venv + uv.lock on first run, installs the project, runs the command
ls -a

Using CPython 3.15.0 interpreter at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f

4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python
Creating virtual environment at: .

venv


   Building weather @ file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwob

e/weather


      Built weather @ file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwob

e/weather


Installed 1 package in 1ms


Hello from weather!


.
..
.git
.gitignore
.python-version
.venv
README.md
pyproject.toml
src
uv.lock


`uv run` **automatically** keeps everything in sync: before running, it checks `pyproject.toml` → updates `uv.lock` → syncs `.venv`. No activation needed (but `source .venv/bin/activate` still works).

### 4.2 Adding and removing dependencies

In [9]:
%%bash
cd "$LAB/weather"
uv add httpx 2>&1 | tail -4                       # runtime dependency (adds a lower bound, e.g. >=0.28.1)
uv add "rich>=13,<16" 2>&1 | tail -1             # your own constraint
uv add --dev pytest 2>&1 | tail -1               # → [dependency-groups] dev
uv add --group lint ruff 2>&1 | tail -1          # → a custom group
uv add --optional yaml pyyaml 2>&1 | tail -1     # → an extra: [project.optional-dependencies]
cat pyproject.toml

 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 ~ weather==0.1.0 (from file:///private/var/folder

s/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwobe/weather)


 ~ weather==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjw

obe/weather)


 ~ weather==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjw

...

[project]
name = "weather"
version = "0.1.0"
description = "Add your description here"
readme = "REA

DME.md"
requires-python = ">=3.15"
dependencies = [
    "httpx>=0.28.1",
    "rich>=13,<16",
]

[pro

ject.scripts]
weather = "weather:main"

[project.optional-dependencies]
yaml = [
    "pyyaml>=6.0.3"

,
]

[build-system]
requires = ["uv_build>=0.13.3,<0.14.0"]
build-backend = "uv_build"

[dependency-

groups]
dev = [
    "pytest>=9.1.1",
]
lint = [
    "ruff>=0.17.1",
]


### Version bounds: `--bounds`

| `uv add httpx --bounds ...` | Writes |
|---|---|
| `lower` (default) | `httpx>=0.28.1` |
| `major` | `httpx>=0.28.1,<1.0.0` (the first non-zero part is the "major") |
| `minor` | `httpx>=0.28.1,<0.29.0` |
| `exact` | `httpx==0.28.1` |

In [10]:
%%bash
cd "$LAB/weather"
uv add --bounds major idna 2>&1 | tail -1
grep idna pyproject.toml
uv remove idna 2>&1 | tail -1
uv remove --optional yaml pyyaml 2>&1 | tail -1

 ~ weather==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjw

obe/weather)


    "idna>=3.20,<4.0",


 ~ weather==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjw

obe/weather)


 ~ weather==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjw

obe/weather)


### Other dependency sources

```bash
uv add "httpx[http2]"                                   # with extras
uv add "pywin32; sys_platform == 'win32'"               # with a marker
uv add git+https://github.com/encode/httpx --tag 0.28.1 # Git (also --branch / --rev)
uv add ../mylib --editable                              # a local package, editable
uv add -r requirements.txt                              # import an existing requirements file
uv add torch --index pytorch=https://download.pytorch.org/whl/cpu   # a named index
```
Non-registry sources are recorded under `[tool.uv.sources]`, keeping `[project.dependencies]` standard:
```toml
[project]
dependencies = ["mylib", "httpx"]

[tool.uv.sources]
mylib = { path = "../mylib", editable = true }
httpx = { git = "https://github.com/encode/httpx", tag = "0.28.1" }
torch = { index = "pytorch" }

[[tool.uv.index]]
name = "pytorch"
url = "https://download.pytorch.org/whl/cpu"
explicit = true                       # only used for packages that ask for it
```

### 4.3 Seeing the dependency tree

In [11]:
%%bash
cd "$LAB/weather"
uv tree --depth 2
echo "--- who depends on idna? (inverted)"
uv tree --invert --package idna 2>/dev/null | head -8

Resolved 17 packages in 2ms


weather v0.1.0
├── httpx v0.28.1
│   ├── anyio v4.15.1
│   ├── certifi v2026.1

0.19
│   ├── httpcore v1.0.9
│   └── idna v3.20
├── rich v15.0.0
│   ├─

─ markdown-it-py v4.2.0
│   └── pygments v2.21.0
└── pytest v9.1.1 (group: dev)
    

├── iniconfig v2.3.1
    ├── packaging v26.3
    ├── pluggy v1.7.0
    └── py

gments v2.21.0


--- who depends on idna? (inverted)


idna v3.20
├── anyio v4.15.1
│   └── httpx v0.28.1
│       └── weather v0.1.0


└── httpx v0.28.1 (*)
(*) Package tree already displayed


`uv tree --outdated` shows available upgrades, `--universal` shows all platforms, and `--show-sizes` shows download sizes.

### 4.4 The lock file: `uv.lock`

In [12]:
%%bash
cd "$LAB/weather"
head -22 uv.lock
echo "..."
grep -c '^\[\[package\]\]' uv.lock | xargs echo "packages locked:"

version = 1
revision = 5
requires-python = ">=3.15"

[[package]]
name = "anyio"
version = "4.15.1"
s

ource = { registry = "https://pypi.org/simple" }
dependencies = [
    { name = "idna" },
]
sdist = {

 url = "https://files.pythonhosted.org/packages/a9/d2/f4d173e22df740bc37b1db102b386ba719b66e95b0f0d7

51f556b387e6d2/anyio-4.15.1.tar.gz", hash = "sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc

957326e578b82a94", size = 276966, upload-time = "2026-09-05T10:42:39.44Z" }
wheels = [
    { url = "

...

 "https://files.pythonhosted.org/packages/a6/bd/1436c50cc46699de0230fbaacc1d1de7bfec413176099949e9d5

112e88f2/certifi-2026.10.19.tar.gz", hash = "sha256:035c1e85572b141bf1c3a3eff0e33a8eb74917522f255907

5ff8d2fea780a1ae", size = 137598, upload-time = "2026-10-20T02:24:16.604Z" }
wheels = [


...


packages locked: 17


- `uv.lock` is a **universal**, cross-platform lock file: exact versions + hashes + download URLs for **every** platform and Python version your `requires-python` allows.
- ✅ **Commit `uv.lock`** for applications (and usually for libraries too); never edit it by hand.
- It's uv-specific; export to standard formats with `uv export` (below).

| Command | Meaning |
|---|---|
| `uv lock` | create/update the lock to match `pyproject.toml` (keeps existing pins where possible) |
| `uv lock --upgrade` | re-resolve everything to the newest allowed versions |
| `uv lock --upgrade-package httpx` | upgrade just one package |
| `uv lock --check` | fail if the lock is out of date (great in CI) |
| `uv sync --locked` | install, but **error** if the lock would change (CI) |
| `uv sync --frozen` | install **exactly** the lock, without checking it against `pyproject.toml` |

In [13]:
%%bash
cd "$LAB/weather"
uv lock --check && echo "✅ lock file is up to date"
uv lock --upgrade-package httpx 2>&1 | tail -1

Resolved 17 packages in 2ms


✅ lock file is up to date


Resolved 17 packages in 61ms


### 4.5 Syncing the environment

`uv sync` makes `.venv` match the lock file **exactly** (installs missing packages, removes extra ones).

In [14]:
%%bash
cd "$LAB/weather"
uv sync 2>&1 | tail -2                                  # default: project + the "dev" group
uv sync --group lint 2>&1 | tail -2                    # also install the lint group
uv sync --no-dev --dry-run 2>&1 | tail -4              # what a production install would remove

Resolved 17 packages in 2ms
Checked 15 packages in 0.59ms


Installed 1 package in 1ms
 + ruff==0.17.1


 - packaging==26.3
 - pluggy==1.7.0
 - pytest==9.1.1
 - ruff==0.17.1


| Option | Effect |
|---|---|
| `--group NAME` / `--all-groups` / `--only-group NAME` | choose dependency groups |
| `--no-dev` / `--only-dev` | exclude / only the `dev` group |
| `--extra NAME` / `--all-extras` | include extras |
| `--no-install-project` | only dependencies (great for Docker layer caching) |
| `--inexact` | don't remove extraneous packages |
| `--locked` / `--frozen` | see above |
| `--no-editable` | install the project non-editable (for deployment) |
| `--active` | sync into the currently activated venv instead of `.venv` |

Default groups can be configured:
```toml
[tool.uv]
default-groups = ["dev", "lint"]       # or "all"
```

### 4.6 Running commands

In [15]:
%%bash
cd "$LAB/weather"
mkdir -p tests
cat > tests/test_basic.py <<'EOF'
import weather
def test_main(capsys):
    weather.main()
    assert "Hello" in capsys.readouterr().out
EOF
uv run pytest -q -p no:cacheprovider 2>&1 | tail -2
uv run python -c "import httpx, rich; print('httpx', httpx.__version__)"
uv run --with ipython python -c "import IPython; print('ipython available only for this run')" 2>&1 | tail -1

.                                                                        [100%]
1 passed in 0.00s


httpx 0.28.1


ipython available only for this run


Useful `uv run` options: `--with pkg` (temporary extra package), `-m module`, `--env-file .env` (load environment variables), `--no-sync`, `--isolated`, `--package member` (workspaces), `--python 3.13`.

### 4.7 Versioning: `uv version`

In [16]:
%%bash
cd "$LAB/weather"
uv version
uv version --bump minor --no-sync
uv version --bump patch --bump beta --dry-run --no-sync 2>&1
uv version --short

weather 0.1.0


Resolved 17 packages in 3ms


weather 0.1.0 => 0.2.0


weather 0.2.0 => 0.2.1b1


0.2.0


Bump kinds: `major`, `minor`, `patch`, `stable`, `alpha`, `beta`, `rc`, `post`, `dev`; or set a value directly: `uv version 2.0.0`.

### 4.8 Exporting the lock file

In [17]:
%%bash
cd "$LAB/weather"
uv export --no-hashes --no-dev 2>/dev/null | head -14
echo "..."
uv export --format pylock.toml -o pylock.toml 2>&1 | tail -1; head -6 pylock.toml

# This file was autogenerated by uv via the following command:
#    uv export --no-hashes --no-dev
-

e .
anyio==4.15.1
    # via httpx
certifi==2026.10.19
    # via
    #   httpcore
    #   httpx
h11==

0.16.0
    # via httpcore
httpcore==1.0.9
    # via httpx
httpx==0.28.1


...


directory = { path = ".", editable = true }


# This file was autogenerated by uv via the following command:
#    uv export --format pylock.toml -

o pylock.toml
lock-version = "1.0"
created-by = "uv"
requires-python = ">=3.15"



Formats: `requirements.txt` (for pip/Docker/platforms that need it), **`pylock.toml`** (the new standard lock format, PEP 751), `cyclonedx1.5` (an SBOM for security/compliance).

## 5. Tools: `uvx` and `uv tool`

Run CLI tools **without installing them** into your project (like `pipx`, but faster):

In [18]:
%%bash
cd "$LAB"
uvx ruff --version 2>&1 | tail -1                         # uvx == uv tool run
uvx --from httpie http --version 2>&1 | tail -1           # when the command name differs from the package
uvx pycowsay@0.0.0.2 "uv is fast" 2>&1 | tail -8          # a specific version

ruff 0.17.1


3.2.4


< uv is fast >
  ----------
   \   ^__^
    \  (oo)\_______
       (__)\       )\/\
           ||---

-w |
           ||     ||



In [19]:
%%bash
uv tool install ruff 2>&1 | tail -2           # installs into an isolated env, puts `ruff` on PATH
uv tool list
uv tool upgrade ruff 2>&1 | tail -1
uv tool uninstall ruff 2>&1 | tail -1
uv tool dir

Installed 1 executable: ruff

obe/.uv-bin` is not on your PATH. To use installed tools, run `export PATH="/var/folders/96/f6xk04z9

1_gdystl3qfqv33r0000gn/T/uv-lab-ealjwobe/.uv-bin:$PATH"` or `uv tool update-shell`.


ruff v0.17.1
- ruff


Nothing to upgrade


Uninstalled 1 executable: ruff


/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwobe/.uv-tools


`uv tool update-shell` adds the tool bin directory to your PATH; `uv tool audit` checks installed tools for vulnerabilities.

> 💡 Project tools (pytest, ruff in a project) belong in dependency groups and run with `uv run`. Global personal tools use `uv tool install`.

## 6. The pip-compatible interface

Prefer project commands, but uv also offers a **drop-in, much faster replacement** for pip/venv/pip-tools workflows:

> ⚠️ **Which environment does `uv pip` modify?** In order: `--python`/`UV_PYTHON` if given, else the **activated** venv (`VIRTUAL_ENV`), else a `.venv` in the current or parent folder. `uv pip sync` **removes everything not listed**, so pointed at the wrong environment it can wipe it, for example the one running your Jupyter kernel. Be explicit with `--python .venv` when in doubt. (Our sandbox set `UV_PYTHON` to the notebook's interpreter, so this cell unsets it first.)

In [20]:
%%bash
unset UV_PYTHON                                            # let uv pip target the local .venv, not the notebook's Python
cd "$LAB" && mkdir legacy && cd legacy
uv venv .venv --python "$(python -c 'import sys; print(sys.executable)')" 2>&1 | head -2   # like python -m venv (add --seed for pip)
printf "requests\nrich>=13\n" > requirements.in
uv pip compile requirements.in -o requirements.txt 2>&1 | tail -1    # pin everything (pip-tools style)
head -12 requirements.txt
uv pip sync --python .venv requirements.txt 2>&1 | tail -1           # make .venv match EXACTLY
uv pip list --python .venv | head -6
uv pip tree --python .venv | head -6
uv pip check --python .venv

Using CPython 3.15.0 interpreter at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f

4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python
Creating virtual environment at: .

venv


    # via requests


# This file was autogenerated by uv via the following command:
#    uv pip compile requirements.in -

...

lizer 3.5.2
idna               3.20
markdown-it-py     4.2.0


requests v2.34.2
├── certifi v2026.10.19
├── charset-normalizer v3.5.2
├── idna v3

.20
└── urllib3 v2.8.0
rich v15.0.0


Checked 9 packages in 0.24ms


All installed packages are compatible


| pip / pip-tools | uv |
|---|---|
| `python -m venv .venv` | `uv venv` |
| `pip install x` | `uv pip install x` |
| `pip install -r req.txt` | `uv pip install -r req.txt` |
| `pip-compile req.in` | `uv pip compile req.in -o req.txt` (`--universal` for all platforms, `--generate-hashes`) |
| `pip-sync req.txt` | `uv pip sync req.txt` |
| `pip freeze`, `pip list`, `pip show` | `uv pip freeze / list / show` |
| `pipdeptree` | `uv pip tree` |

## 7. Workspaces (monorepos)

A **workspace** is several packages developed together, sharing **one** lock file and one `.venv`.

In [21]:
%%bash
cd "$LAB" && uv init --bare --author-from none shop >/dev/null 2>&1 && cd shop
cat >> pyproject.toml <<'EOF'

[tool.uv.workspace]
members = ["packages/*"]
EOF
mkdir packages
uv init --lib --author-from none --vcs none packages/core >/dev/null 2>&1
uv init --lib --author-from none --vcs none packages/api  >/dev/null 2>&1
cd packages/api && uv add core 2>&1 | tail -1 && cd ../..        # depend on a sibling member
sed -n '/\[project\]/,/^$/p;/tool.uv.sources/,$p' packages/api/pyproject.toml
uv workspace list 2>&1
uv run --package api python -c "import core, api; print('workspace imports work')"

 + core==0.1.0 (from file:///private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwobe

/shop/packages/core)


[project]
name = "api"
version = "0.1.0"
description = "Add your description here"
readme = "README.

md"
requires-python = ">=3.15"
dependencies = [
    "core",
]

[tool.uv.sources]
core = { workspace 

= true }


api
core
shop


workspace imports work


- `[tool.uv.sources] core = { workspace = true }` links members to each other (editable).
- `uv sync --all-packages`, `uv run --package api ...`, `uv build --all-packages`.

## 8. Code quality built in (🧪 preview commands)

> In uv 0.13, `uv format`, `uv check` and `uv audit` are **preview features**: they work, but print an *experimental* warning and may change. Silence it with `--preview-features format-command` (etc.) or set `UV_PREVIEW=1`.

In [22]:
%%bash
cd "$LAB/weather"
printf 'import os\ndef add( a,b ):\n    return a+b\n' > src/weather/messy.py
uv format --diff 2>&1 | head -12            # Ruff formatter (downloaded on demand)
uv format 2>&1 | tail -1
cat src/weather/messy.py

-command` to disable this warning.
 Downloaded ruff v0.15.22
--- 

src/weather/messy.py
+++ src/weather/messy.py
@@ -1,3 +1,5 @@
 import os
-def add( a,b ):
-    retur

n a+b
+
+
+def add(a, b):


2 files reformatted, 1 file left unchanged


import os


def add(a, b):
    return a + b


- `uv format` runs **Ruff's formatter** (`--check`, `--diff`, `-- <ruff args>`).
- `uv check` runs **ty**, Astral's fast type checker, on the project (`--fix` applies safe fixes).
- `uv audit` checks your locked dependencies against the OSV vulnerability database (`--output-format json|sarif`, `--ignore ID`).

In [23]:
%%bash
cd "$LAB/weather"
printf 'def double(x: int) -> int:\n    return x * 2\n\ndouble("oops")\n' > src/weather/typing_demo.py
uv check 2>&1 | grep -E "error|warning|Found|All checks" | head -6
echo "--- audit:"
uv audit 2>&1 | tail -3

ommand` to disable this warning.
error[invalid-argument-type]: Argument to function `double` is inco

rrect
Found 1 diagnostic


--- audit:


ommand` to disable this warning.
Resolved 17 packages in 3ms
Found no known vulnerabilities and no a

dverse project statuses in 16 packages


## 9. Building and publishing

```bash
uv build                 # → dist/weather-0.2.0.tar.gz and dist/weather-0.2.0-py3-none-any.whl
uv build --wheel         # only a wheel (or --sdist)
uv publish               # upload dist/* to PyPI (token / trusted publishing)
uv publish --index testpypi
```

In [24]:
%%bash
cd "$LAB/weather"
uv build 2>&1 | tail -2
ls dist

Successfully built dist/weather-0.2.0.tar.gz
Successfully built dist/weather-0.2.0-py3-none-any.whl


weather-0.2.0-py3-none-any.whl
weather-0.2.0.tar.gz


Lesson 06 explains what's inside these files; lesson 07 covers publishing (including a live upload to a local package index).

## 10. Configuration

uv reads settings from (highest priority first): command-line flags → environment variables (`UV_*`) → `[tool.uv]` in `pyproject.toml` → `uv.toml` in the project → user config `~/.config/uv/uv.toml` → system config.

```toml
[tool.uv]
default-groups = ["dev"]
required-version = ">=0.13"           # fail fast if a teammate's uv is too old
exclude-newer = "2026-10-01T00:00:00Z" # resolve as if it were this date (reproducible / supply-chain safety)
resolution = "highest"                 # or "lowest-direct" to test your minimum versions
prerelease = "if-necessary-or-explicit"
compile-bytecode = true                # faster first import in production

[[tool.uv.index]]
name = "company"
url = "https://pypi.company.internal/simple"
default = true                         # replace PyPI
```

Handy environment variables: `UV_CACHE_DIR`, `UV_PYTHON`, `UV_OFFLINE=1`, `UV_NO_SYNC`, `UV_FROZEN`, `UV_LOCKED`, `UV_INDEX`, `UV_DEFAULT_INDEX`, `UV_PUBLISH_TOKEN`, `UV_COMPILE_BYTECODE`, `UV_LINK_MODE=copy`.

> ⚠️ **Deprecated in uv:** `--index-url` / `--extra-index-url` (use `--default-index` / `--index`), and `[tool.uv] dev-dependencies = [...]` (use `[dependency-groups]`).

### Authentication & private indexes

```bash
uv auth login pypi.company.internal     # store credentials (uv auth logout / token / dir)
export UV_INDEX_COMPANY_USERNAME=me UV_INDEX_COMPANY_PASSWORD=secret   # per-index env credentials
```

### The cache

In [25]:
%%bash
uv cache dir
uv cache size 2>/dev/null || true
uv cache prune 2>&1 | tail -1

/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/uv-lab-ealjwobe/.uv-cache


263647232


Removed 8970 files (94.3MiB)


uv hard-links/copies files from a global cache, which is why re-creating environments takes milliseconds. `uv cache clean` clears it; `--offline` works purely from the cache.

## 11. uv in Docker and CI

**Dockerfile** (multi-stage friendly, with layer caching):
```dockerfile
FROM python:3.15-slim
COPY --from=ghcr.io/astral-sh/uv:0.13 /uv /uvx /bin/
WORKDIR /app
ENV UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy
# 1) dependencies only (cached until the lock file changes)
COPY pyproject.toml uv.lock ./
RUN uv sync --locked --no-install-project --no-dev
# 2) then the project itself
COPY . .
RUN uv sync --locked --no-dev
CMD ["uv", "run", "--no-sync", "weather"]
```

**GitHub Actions**:
```yaml
- uses: actions/checkout@v5
- uses: astral-sh/setup-uv@v7
  with:
    enable-cache: true
- run: uv sync --locked --all-groups
- run: uv run ruff check . && uv run pytest
```

## 12. Migrating to uv

| From | How |
|---|---|
| `requirements.txt` | `uv init --bare` then `uv add -r requirements.txt` |
| pip-tools | `uv pip compile` / `uv pip sync` (same files), or move to a project |
| Poetry / Pipenv / PDM | `uvx migrate-to-uv` (a community tool that converts `pyproject.toml` and locks) |
| pipx | `uv tool install` |
| pyenv | `uv python install` + `.python-version` |

## 13. Cheat-sheet

```bash
uv init myapp && cd myapp        # new project
uv add httpx                     # dependency
uv add --dev pytest              # dev dependency
uv run python main.py            # run (auto-syncs)
uv run pytest                    # run tools in the project env
uv lock --upgrade                # upgrade everything
uv sync --locked                 # CI / teammates: install exactly the lock
uv tree --outdated               # what can be upgraded
uvx ruff check .                 # one-off tool
uv tool install httpie           # global tool
uv run --with rich script.py     # quick script with a dependency
uv python install 3.15           # get a Python
uv build && uv publish           # ship it
```

## 📝 Summary
- uv manages **Python versions, environments, dependencies, lock files, scripts, tools, builds and publishing**, very fast.
- Projects: `init → add → run/sync → lock`; commit `pyproject.toml` + `uv.lock`; CI uses `--locked`.
- Scripts carry their own dependencies via PEP 723; `uvx` runs tools on demand.
- The pip interface (`uv pip ...`) speeds up legacy workflows without changing them.
- Workspaces, `uv format`, `uv check`, `uv audit`, `uv build`, `uv publish` complete the toolbox.

## 🏋️ Practice
1. `uv init --app todo`, add `rich` and a dev group with `pytest`, write a test, and run it with `uv run pytest`.
2. Upgrade only `rich` with `uv lock --upgrade-package rich`, then check `uv tree --outdated`.
3. Turn a single file into a self-contained script with `uv init --script` + `uv add --script`, add the shebang and run it directly.
4. Create a workspace with a `core` library and a `cli` app that depends on it.
5. Export `requirements.txt` without dev dependencies for a deployment platform.
6. Write a Dockerfile for your project using `uv sync --locked --no-install-project` for layer caching.